# Cyclistic — Análisis de patrones de uso

## Objetivo

Este notebook corresponde a la etapa de análisis del caso de estudio Cyclistic, desarrollado como trabajo final del Google Data Analytics Professional Certificate.

A partir del conjunto de datos preparado y depurado en la etapa anterior, se busca identificar diferencias en los patrones de uso de bicicletas compartidas entre miembros con suscripción anual (`member`) y usuarios casuales (`casual`).

## Datos utilizados

Se utiliza el archivo consolidado de viajes correspondiente al período de septiembre de 2025 a agosto de 2026.

**Entorno de trabajo:** este notebook fue desarrollado en Google Colab utilizando archivos almacenados en Google Drive. Las rutas corresponden al entorno original de trabajo y deben adaptarse para ejecutar el análisis en otro entorno.

## Análisis realizados

- Distribución de viajes por tipo de usuario.
- Comparación de la duración de los viajes, incluyendo y excluyendo valores extremos.
- Patrones de uso según el día de la semana.
- Distribución horaria y diferencias entre días laborables y fines de semana.
- Evolución mensual de los viajes.
- Preferencias de tipos de bicicleta según la membresía.

**Propósito:** identificar patrones de comportamiento que permitan comprender las diferencias entre ambos grupos y aportar información para las recomendaciones finales del caso de estudio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
ruta = "/content/drive/MyDrive/Colab Notebooks/cyclistic/datos_procesados/cyclistic_202509_202608_limpio.csv"

cyclistic = pd.read_csv(
    ruta,
    parse_dates=["started_at", "ended_at"]
)

print("Registros:", len(cyclistic))
print("Columnas:", len(cyclistic.columns))

Registros: 6115947
Columnas: 19


## Distribución de viajes y duración según el tipo de usuario

En primer lugar, se analiza la cantidad y proporción de viajes realizados por miembros anuales (`member`) y usuarios casuales (`casual`).

Posteriormente, se compara la duración de los viajes de ambos grupos utilizando la media y la mediana, con el objetivo de identificar diferencias en sus comportamientos.

### Análisis de valores extremos

Como parte de la preparación de los datos, se identificaron como viajes "extremos" a aquellos con duraciones inferiores a 1 minuto o superiores a 120 minutos.

Para evaluar su influencia sobre los resultados, se comparan las duraciones incluyendo todos los registros y excluyendo aquellos identificados como extremos.

También se calcula la proporción de viajes extremos por tipo de usuario, diferenciando los viajes de duración muy corta y muy larga.

Este procedimiento permite evaluar la sensibilidad de las estadísticas a los valores extremos sin eliminarlos definitivamente del conjunto de datos y poder considerar dos escenarios a la hora de visualizar los datos.

In [ ]:
viajes_por_usuario = cyclistic["member_casual"].value_counts()

porcentaje_por_usuario = (
    cyclistic["member_casual"].value_counts(normalize=True) * 100
).round(2)

resumen_usuarios = pd.DataFrame({
    "viajes": viajes_por_usuario,
    "porcentaje": porcentaje_por_usuario
})

display(resumen_usuarios)

,viajes,porcentaje
member_casual,,
member,3955594,64.68
casual,2160353,35.32


In [ ]:
resumen_duracion = (
    cyclistic
    .groupby("member_casual")["ride_length_min"]
    .agg(["count", "mean", "median"])
    .round(2)
)

display(resumen_duracion)

,count,mean,median
member_casual,,,
casual,2160353,20.70,10.86
member,3955594,12.37,8.55


In [ ]:
duracion_sin_extremos = (
    cyclistic[cyclistic["extremo"] == 0]
    .groupby("member_casual")["ride_length_min"]
    .agg(["count", "mean", "median"])
    .round(2)
)

display(duracion_sin_extremos)

,count,mean,median
member_casual,,,
casual,2063564,16.42,11.15
member,3863475,11.63,8.71


In [ ]:
porcentaje_extremos = (
    cyclistic.groupby("member_casual")["extremo"]
    .mean()
    .mul(100)
    .round(2)
)

display(porcentaje_extremos)

,extremo
member_casual,
casual,4.48
member,2.33


In [ ]:
resumen_extremos = cyclistic.groupby("member_casual").agg(
    viajes_menores_1min=("ride_length_min", lambda x: (x < 1).sum()),
    viajes_mayores_120min=("ride_length_min", lambda x: (x > 120).sum())
)

display(resumen_extremos)

,viajes_menores_1min,viajes_mayores_120min
member_casual,,
casual,75082,21707
member,83793,8326


## Patrones de uso según el día de la semana y la hora

En esta etapa se analiza cómo se distribuyen los viajes de miembros anuales y usuarios casuales a lo largo de la semana y en distintos horarios del día.

### Distribución semanal

Se calcula la cantidad y proporción de viajes realizados cada día de la semana para ambos grupos.

Además, se considera la cantidad de veces que aparece cada día dentro del período estudiado, con el objetivo de obtener promedios diarios que permitan realizar comparaciones más adecuadas.

### Distribución horaria

Se analiza la cantidad de viajes según la hora de inicio, diferenciando entre miembros anuales y usuarios casuales.

Posteriormente, se clasifican los días en dos categorías: **laborables** (lunes a viernes) y **fines de semana** (sábado y domingo).

Esta clasificación permite comparar los patrones horarios de ambos grupos y explorar posibles diferencias entre usos asociados a desplazamientos cotidianos y actividades recreativas.

Finalmente, se calculan promedios por tipo de día para evitar que la distinta cantidad de días laborables y fines de semana distorsione las comparaciones.

In [ ]:
cyclistic["dia_semana"] = cyclistic["started_at"].dt.dayofweek

In [ ]:
viajes_por_dia = (
    cyclistic
    .groupby(["dia_semana", "member_casual"])
    .size()
    .unstack()
)

display(viajes_por_dia)

member_casual,casual,member
dia_semana,,
0,257881,566150
1,246857,630934
2,247902,631953
3,268025,625494
4,334557,577940
5,456663,502764
6,348468,420359


In [ ]:
viajes_por_dia["casual_%"] = (
    viajes_por_dia["casual"] / viajes_por_dia["casual"].sum() * 100
).round(2)

viajes_por_dia["member_%"] = (
    viajes_por_dia["member"] / viajes_por_dia["member"].sum() * 100
).round(2)

display(viajes_por_dia)

member_casual,casual,member,casual_%,member_%
dia_semana,,,,
0,257881,566150,11.94,14.31
1,246857,630934,11.43,15.95
2,247902,631953,11.48,15.98
3,268025,625494,12.41,15.81
4,334557,577940,15.49,14.61
5,456663,502764,21.14,12.71
6,348468,420359,16.13,10.63


In [ ]:
# Cantidad de veces que aparece cada día en el período
fechas = pd.date_range("2025-09-01", "2026-08-31")
cantidad_dias = fechas.dayofweek.value_counts().sort_index()

# Promedio diario por tipo de usuario
promedio_por_dia = (
    viajes_por_dia[["casual", "member"]]
    .div(cantidad_dias, axis=0)
    .round(1)
)

display(promedio_por_dia)

member_casual,casual,member
dia_semana,,
0,4865.7,10682.1
1,4747.2,12133.3
2,4767.3,12152.9
3,5154.3,12028.7
4,6433.8,11114.2
5,8782.0,9668.5
6,6701.3,8083.8


In [ ]:
# Extraer la hora de inicio de cada viaje
cyclistic["hora_inicio"] = cyclistic["started_at"].dt.hour

# Contar viajes por hora y tipo de usuario
viajes_por_hora = (
    cyclistic
    .groupby(["hora_inicio", "member_casual"])
    .size()
    .unstack()
)

display(viajes_por_hora)

member_casual,casual,member
hora_inicio,,
0,46730,37307
1,30692,22734
2,19410,13418
3,11094,8747
4,8863,9853
5,13901,38464
6,29744,112210
7,53705,222028
8,76939,287997


In [ ]:
cyclistic["tipo_dia"] = cyclistic["dia_semana"].apply(
    lambda dia: "Fin de semana" if dia >= 5 else "Laborable"
)

In [ ]:
viajes_hora_tipo_dia = (
    cyclistic
    .groupby(["member_casual", "tipo_dia", "hora_inicio"])
    .size()
    .reset_index(name="viajes")
)

display(viajes_hora_tipo_dia.head(10))

,member_casual,tipo_dia,hora_inicio,viajes
0,casual,Fin de semana,0,26236
1,casual,Fin de semana,1,18676
2,casual,Fin de semana,2,12127
3,casual,Fin de semana,3,6676
4,casual,Fin de semana,4,4085
5,casual,Fin de semana,5,3645
6,casual,Fin de semana,6,5873
7,casual,Fin de semana,7,9532
8,casual,Fin de semana,8,17172
9,casual,Fin de semana,9,28618


In [ ]:
# Tabla de horarios para usuarios casuales
horarios_casual = (
    viajes_hora_tipo_dia[
        viajes_hora_tipo_dia["member_casual"] == "casual"
    ]
    .pivot(
        index="hora_inicio",
        columns="tipo_dia",
        values="viajes"
    )
)

display(horarios_casual)

tipo_dia,Fin de semana,Laborable
hora_inicio,,
0,26236,20494
1,18676,12016
2,12127,7283
3,6676,4418
4,4085,4778
5,3645,10256
6,5873,23871
7,9532,44173
8,17172,59767


In [ ]:
# Tabla de horarios para miembros anuales
horarios_member = (
    viajes_hora_tipo_dia[
        viajes_hora_tipo_dia["member_casual"] == "member"
    ]
    .pivot(
        index="hora_inicio",
        columns="tipo_dia",
        values="viajes"
    )
)

display(horarios_member)

tipo_dia,Fin de semana,Laborable
hora_inicio,,
0,19912,17395
1,13545,9189
2,8271,5147
3,4709,4038
4,3035,6818
5,4470,33994
6,9823,102387
7,16724,205304
8,29964,258033


In [ ]:
promedio_horarios_casual = horarios_casual.copy()

promedio_horarios_casual["Laborable"] /= 261
promedio_horarios_casual["Fin de semana"] /= 104

display(promedio_horarios_casual.round(1))

tipo_dia,Fin de semana,Laborable
hora_inicio,,
0,252.3,78.5
1,179.6,46.0
2,116.6,27.9
3,64.2,16.9
4,39.3,18.3
5,35.0,39.3
6,56.5,91.5
7,91.7,169.2
8,165.1,229.0


In [ ]:
promedio_horarios_member = horarios_member.copy()

promedio_horarios_member["Laborable"] /= 261
promedio_horarios_member["Fin de semana"] /= 104

display(promedio_horarios_member.round(1))

tipo_dia,Fin de semana,Laborable
hora_inicio,,
0,191.5,66.6
1,130.2,35.2
2,79.5,19.7
3,45.3,15.5
4,29.2,26.1
5,43.0,130.2
6,94.5,392.3
7,160.8,786.6
8,288.1,988.6


## Evolución mensual y estacionalidad

En esta etapa se analiza la evolución de la cantidad de viajes a lo largo de los 12 meses del período estudiado, diferenciando entre miembros anuales (`member`) y usuarios casuales (`casual`).

Se agrupan los registros según el mes de inicio del viaje para identificar variaciones en la demanda y comparar el comportamiento de ambos grupos.

Este análisis permite observar posibles patrones estacionales, como aumentos o disminuciones en el uso de bicicletas durante determinadas épocas del año.

También permite evaluar si las variaciones mensuales afectan de manera similar a ambos grupos o si alguno presenta una mayor sensibilidad a los cambios de temporada.

**Interpretación:** las variaciones mensuales observadas sugieren que las estaciones del año influyen en los patrones de uso de bicicletas. Este factor podría tener un impacto diferente sobre los miembros anuales y los usuarios casuales.

In [ ]:
# Crear una variable de año y mes
cyclistic["mes"] = cyclistic["started_at"].dt.to_period("M")

In [ ]:
inicio = pd.Timestamp("2025-09-01")
fin = pd.Timestamp("2026-09-01")

cyclistic_periodo = cyclistic[
    (cyclistic["started_at"] >= inicio) &
    (cyclistic["started_at"] < fin)
].copy()

print("Viajes originales:", len(cyclistic))
print("Viajes dentro del período:", len(cyclistic_periodo))
print("Viajes fuera del período:", len(cyclistic) - len(cyclistic_periodo))

Viajes originales: 6115947
Viajes dentro del período: 6115679
Viajes fuera del período: 268


In [ ]:
viajes_por_mes = (
    cyclistic_periodo
    .groupby([
        cyclistic_periodo["started_at"].dt.to_period("M"),
        "member_casual"
    ])
    .size()
    .unstack(fill_value=0)
)

display(viajes_por_mes)

member_casual,casual,member
started_at,,
2025-09,265268,449294
2025-10,224038,422058
2025-11,99091,257421
2025-12,28090,112455
2026-01,24740,113042
2026-02,41173,160281
2026-03,87850,229165
2026-04,131514,316738
2026-05,245143,408604


In [ ]:
participacion_mensual = viajes_por_mes.copy()

participacion_mensual["total"] = (
    participacion_mensual["casual"] + participacion_mensual["member"]
)

participacion_mensual["casual_%"] = (
    participacion_mensual["casual"] /
    participacion_mensual["total"] * 100
).round(2)

participacion_mensual["member_%"] = (
    participacion_mensual["member"] /
    participacion_mensual["total"] * 100
).round(2)

display(participacion_mensual)

member_casual,casual,member,total,casual_%,member_%
started_at,,,,,
2025-09,265268,449294,714562,37.12,62.88
2025-10,224038,422058,646096,34.68,65.32
2025-11,99091,257421,356512,27.79,72.21
2025-12,28090,112455,140545,19.99,80.01
2026-01,24740,113042,137782,17.96,82.04
2026-02,41173,160281,201454,20.44,79.56
2026-03,87850,229165,317015,27.71,72.29
2026-04,131514,316738,448252,29.34,70.66
2026-05,245143,408604,653747,37.50,62.50


## Preferencias según el tipo de bicicleta

En esta etapa se analiza la distribución de los viajes según el tipo de bicicleta utilizado (`rideable_type`), diferenciando entre miembros anuales (`member`) y usuarios casuales (`casual`).

Se comparan las cantidades y proporciones de viajes realizados con cada tipo de bicicleta, buscando identificar diferencias en las preferencias de ambos grupos.

Este análisis complementa los patrones de duración, frecuencia y horarios estudiados anteriormente, y permite construir una interpretación más completa de los comportamientos de uso de Cyclistic.

In [ ]:
# Identificar tipos de bicicleta disponibles
cyclistic_periodo["rideable_type"].value_counts()

,count
rideable_type,
electric_bike,4281995
classic_bike,1833684


In [ ]:
viajes_por_bicicleta = (
    cyclistic_periodo
    .groupby(["rideable_type", "member_casual"])
    .size()
    .unstack(fill_value=0)
)

display(viajes_por_bicicleta)

member_casual,casual,member
rideable_type,,
classic_bike,570174,1263510
electric_bike,1589951,2692044


In [ ]:
porcentaje_bicicletas = (
    viajes_por_bicicleta
    .div(viajes_por_bicicleta.sum(axis=0), axis=1)
    .mul(100)
    .round(2)
)

display(porcentaje_bicicletas)

member_casual,casual,member
rideable_type,,
classic_bike,26.4,31.94
electric_bike,73.6,68.06


In [ ]:
print("Registros originales:", len(cyclistic))
print("Registros analizados:", len(cyclistic_periodo))
print("Registros excluidos:", len(cyclistic) - len(cyclistic_periodo))

print("\nPrimera fecha:", cyclistic_periodo["started_at"].min())
print("Última fecha:", cyclistic_periodo["started_at"].max())

print("\nRide ID duplicados:", cyclistic_periodo["ride_id"].duplicated().sum())

Registros originales: 6115947
Registros analizados: 6115679
Registros excluidos: 268

Primera fecha: 2025-09-01 00:00:03.776000
Última fecha: 2026-08-31 23:55:36.979000

Ride ID duplicados: 0


**Nota metodológica:** los análisis exploratorios iniciales se realizaron sobre el conjunto consolidado de viajes. Posteriormente, se aplicó un filtro para conservar exclusivamente los registros comprendidos entre septiembre de 2025 y agosto de 2026, excluyendo 268 viajes fuera del período. Los resúmenes finales se calcularon sobre este conjunto filtrado.

## Controles finales y resumen del análisis

Para finalizar, se verifica la cantidad de registros incluidos en el período de estudio, comprendido entre septiembre de 2025 y agosto de 2026.

Se calcula un resumen final por tipo de usuario, incluyendo la cantidad de viajes y las estadísticas de duración, tanto con valores extremos como sin ellos.

Estos resultados permiten disponer de una referencia consolidada para contrastar los principales hallazgos del análisis y preparar las visualizaciones posteriores.

In [ ]:
resumen_final = (
    cyclistic_periodo
    .groupby("member_casual")
    .agg(
        viajes=("ride_id", "count"),
        duracion_media=("ride_length_min", "mean"),
        duracion_mediana=("ride_length_min", "median"),
        porcentaje_extremos=("extremo", "mean")
    )
)

resumen_final["participacion_%"] = (
    resumen_final["viajes"] /
    resumen_final["viajes"].sum() * 100
)

resumen_final["porcentaje_extremos"] *= 100

display(resumen_final.round(2))

,viajes,duracion_media,duracion_mediana,porcentaje_extremos,participacion_%
member_casual,,,,,
casual,2160125,20.66,10.86,4.48,35.32
member,3955554,12.37,8.55,2.33,64.68


In [ ]:
duracion_final_sin_extremos = (
    cyclistic_periodo[cyclistic_periodo["extremo"] == 0]
    .groupby("member_casual")["ride_length_min"]
    .agg(["count", "mean", "median"])
    .round(2)
)

display(duracion_final_sin_extremos)

,count,mean,median
member_casual,,,
casual,2063402,16.42,11.15
member,3863438,11.63,8.71
